# Klasifikasi Status Kelangsungan Hidup Pasien Operasi pada Dataset Haberman

**Kelompok 4**
- Mohammad Zaydan Alrafi (103032400015).
- Mohammad Bagus Satrio (103032400099).

**Algoritma yang digunakan:** Decision Tree (Rule-based) vs K-Nearest Neighbors (Distance-based) vs Naive Bayes (Probabilistic-based).

## 1. Pendahuluan & Pemaparan Data
Dalam tugas besar ini, kelompok kami menganalisis **Dataset Kelangsungan Hidup Haberman (Haberman's Survival Dataset)**. Dataset ini merupakan salah satu dataset medis klasik dan autentik yang dipublikasikan secara luas di bidang pemelajaran mesin melalui repositori akademis **UCI Machine Learning Repository**.

### Asal-Usul dan Keaslian Data:
1. **Sumber Data**: Data ini dikumpulkan dari studi klinis riil yang dilakukan di **Billings Hospital, University of Chicago**, oleh Dr. J.R. Haberman.
2. **Rentang Waktu**: Studi mencakup data pasien kanker payudara yang telah menjalani operasi bedah dalam kurun waktu **1958 hingga 1970**.
3. **Kredibilitas Akademis**: Sebagai dataset medis historis yang berakar pada catatan klinis nyata rumah sakit universitas, data ini memiliki tingkat keaslian (*authenticity*) 100% dan bebas dari rekayasa sintetis.

### Detail Atribut & Klinis:
Dataset ini memuat data sebanyak **306 pasien** dengan 3 fitur klinis (fitur independen) dan 1 label target (fitur dependen):
1. **Usia Pasien saat Operasi (`age`)**: Merepresentasikan kondisi fisiologis umum pasien. Usia biologis memengaruhi kekuatan sistem imun, kemampuan pemulihan seluler pasca-operasi bedah, serta keberadaan komorbiditas lain.
2. **Tahun Operasi (`op_Year`)**: Direpresentasikan dalam format dua angka terakhir (misal: 64 berarti tahun 1964). Atribut ini penting karena menangkap perkembangan historis dunia medis pada masa itu—seperti sterilisasi, penemuan antibiotik baru, teknik pembiusan, serta standar kebersihan ruang operasi yang meningkat secara bertahap dari tahun 1958 ke 1970.
3. **Jumlah Kelenjar Getah Bening Ketiak Positif (`axil_nodes`)**: Merupakan indikator klinis utama untuk mendeteksi metastasis (penyebaran kanker) ke sistem limfatik regional. Jumlah kelenjar getah bening ketiak yang positif mengandung sel kanker menunjukkan stadium keganasan kanker yang lebih lanjut.
4. **Status Kelangsungan Hidup (`surv_status` / Target)**: 
   - Kelas `1` (dalam kode dipetakan menjadi `0`): Pasien bertahan hidup 5 tahun atau lebih pasca-operasi (kondisi prognostik baik).
   - Kelas `2` (dalam kode dipetakan menjadi `1`): Pasien meninggal dalam waktu kurang dari 5 tahun pasca-operasi (kondisi prognostik buruk / kelas kritis).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('haberman.csv')
df.columns = ['age', 'op_Year', 'axil_nodes', 'surv_status']

print("tampilkan baris pertama:")
display(df.head())

print("\nstatus kelangsungan hidup:")
print("1 = bertahan hidup 5 tahun atau lebih\n2 = meninggal dalam 5 tahun")
print(df['surv_status'].value_counts())


plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='surv_status', palette='Set2')
plt.title('distribusi kelas (survival status)')
plt.xlabel('Status (1 = >= 5 tahun, 2 = < 5 tahun)')
plt.ylabel('jumlah Pasien')
plt.show()

**Catatan:**
Dari visualisasi di atas, kelihatan banget kalau data kita tidak seimbang (imbalanced). Pasien yang bertahan hidup lama (Kelas 1) ada 225 orang, sedangkan yang meninggal cepat (Kelas 2) cuma 81 orang (rasionya hampir 3:1). 

Kalau data timpang gini langsung dipake buat ngelatih model, modelnya bakal kurang maksimal dan cenderung nebak Kelas 1 terus karena peluang awalnya gede. Masalah ini bakal kita beresin di tahap pra-pemrosesan nanti biar modelnya lebih adil mendeteksi pasien yang berisiko (Kelas 2).

## 2. Pra-Pemrosesan Data
Sebelum melatih model, dataset dibagi menjadi training set (80%) dan testing set (20%) secara acak namun terkontrol. Selanjutnya, fitur-fitur numerik dinormalisasi menggunakan **StandardScaler** untuk menyamakan skala dan variansnya.

### Penggunaan StandardScaler pada Masing-Masing Algoritma:
1. **K-Nearest Neighbors (KNN) [Sangat Wajib]**:
   KNN menentukan klasifikasi berdasarkan jarak terdekat antar-titik data di ruang berdimensi-$N$ menggunakan rumus Jarak Euclidean:
   $$d(x, y) = \sqrt{(x_1 - y_1)^2 + (x_2 - y_2)^2 + (x_3 - y_3)^2}$$
   Jika fitur tidak discaling, fitur dengan rentang nilai nominal yang secara alami lebih besar atau memiliki varians besar akan mendominasi perhitungan jarak. Sebagai contoh, fitur `age` memiliki rentang nominal 30–83 tahun, sedangkan `axil_nodes` sebagian besar berada di rentang 0–5. Selisih numerik pada fitur `age` akan menutupi selisih klinis yang sangat krusial pada `axil_nodes`, sehingga model menjadi bias. Scaling mengubah semua fitur agar memiliki rata-rata ($\mu$) = 0 dan standar deviasi ($\sigma$) = 1.
2. **Decision Tree (Rule-based) [Tidak Berpengaruh secara Matematis]**:
   Decision Tree membagi data (*splitting*) secara hierarkis menggunakan batas ambang (*threshold*) nilai fitur tunggal pada setiap *node* (misalnya: `axil_nodes <= 2`). Keputusan ini bersifat monotonik dan hanya bergantung pada urutan peringkat nilai fitur, bukan pada magnitudo nominalnya. Oleh karena itu, penskalaan fitur tidak mengubah struktur percabangan pohon keputusan. Penskalaan tetap diterapkan agar representasi input seragam di semua model.
3. **Gaussian Naive Bayes (Probabilistic) [Tidak Berpengaruh secara Matematis]**:
   Naive Bayes memodelkan distribusi probabilitas posterior menggunakan fungsi densitas Gaussian:
   $$P(x_i \mid C_k) = \frac{1}{\sqrt{2\pi\sigma_k^2}} e^{-\frac{(x_i - \mu_k)^2}{2\sigma_k^2}}$$
   Penskalaan linier (StandardScaler) mengubah nilai rata-rata ($\mu$) dan varians ($\sigma^2$) secara proporsional. Secara matematis, rasio probabilitas posterior untuk setiap kelas $P(C_k \mid x)$ akan tetap identik, baik sebelum maupun sesudah penskalaan.

### Parameter Pemisahan Data:
- **`test_size=0.2` (Rasio 80:20)**: Mengingat jumlah sampel Haberman yang relatif kecil (306 baris), pembagian ini adalah rasio optimal (*best practice*). Model dibekali data latih yang cukup (244 sampel) untuk mengenali pola klinis, sekaligus menyisakan data uji (62 sampel) yang cukup representatif untuk evaluasi generalisasi model.
- **`stratify=y` [Sangat Krusial]**: Menjamin bahwa proporsi kelas target (pasien selamat vs meninggal) pada data latih dan data uji tetap seimbang dan sama persis dengan rasio dataset asli (sekitar 73.5% : 26.5%). Tanpa stratifikasi pada data kecil, ada risiko salah satu kelas tidak terwakili secara proporsional di data uji, membuat evaluasi menjadi tidak valid.
- **`random_state=42`**: Mengunci seed generator keacakan untuk memastikan hasil pemisahan data selalu konsisten dan dapat direproduksi (*reproducible*) saat eksperimen diulang.

* Nilai test_size=0.2 
berarti data dibagi menjadi 80% untuk data yang kita training dan 20% untuk data uji. Dari total 306 data, sekitar 244 data digunakan untuk melatih model, sedangkan 62 data digunakan untuk menguji hasil prediksi model. Pembagian 80:20 sering digunakan karena memberikan jumlah data latih yang cukup sekaligus menyediakan data uji yang memadai untuk mengecek kinerja model.

* Penjelasan stratify=y
Parameter stratify=y digunakan agar perbandingan jumlah kelas pada data training dan data uji tetap sama seperti pada dataset asli. Pada dataset Haberman, jumlah pasien pada Kelas 1 lebih banyak dibandingkan Kelas 2. Dengan stratifikasi, kedua kelompok data tetap memiliki proporsi kelas yang seimbang. Hal ini membuat hasil pengujian model menjadi lebih adil dan lebih dapat dipercaya.

* Penjelasan random_state=42
Parameter random_state=42 digunakan untuk mengunci proses pembagian data agar hasilnya selalu sama setiap kali program dijalankan. Dengan begitu, kelompok kami akan mendapatkan pembagian data yang sama sehingga hasil percobaan dapat dibandingkan dengan mudah dan tetap konsisten.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('surv_status', axis=1)
y = df['surv_status'].map({1: 0, 2: 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dimensi X_train sebelum scaling:", X_train.shape)
print("Dimensi X_test:", X_test.shape)

## 3. Penanganan Data Timpang (SMOTE)
Visualisasi distribusi kelas menunjukkan ketidakseimbangan kelas yang mencolok (225 pasien selamat vs 81 meninggal, rasio ~3:1). Ketimpangan kelas ini diatasi menggunakan **SMOTE (Synthetic Minority Over-sampling Technique)** pada data latih.

###SMOTE & Bahaya Accuracy Paradox:
Jika kita langsung melatih model pada data latih yang timpang, model cenderung mengalami *Accuracy Paradox*. Model akan sangat mudah menebak kelas mayoritas (pasien selamat) karena peluang awalnya (*prior probability*) jauh lebih besar. Meskipun akurasi global terlihat tinggi, model akan gagal total dalam mendeteksi pasien berisiko tinggi (kelas minoritas), yang secara medis sangat berbahaya.

### Cara Kerja Algoritma SMOTE secara Detail:
1. SMOTE menganalisis ruang fitur untuk sampel-sampel di kelas minoritas (pasien meninggal / Kelas 1 dalam representasi baru).
2. Untuk setiap sampel minoritas, algoritma mencari $k$-tetangga terdekatnya yang juga berada di kelas minoritas (biasanya $k=5$).
3. SMOTE menarik garis imajiner antar-tetangga tersebut di ruang fitur berdimensi-$N$.
4. Titik data sintetis baru dibuat dengan cara memilih titik acak di sepanjang garis tersebut menggunakan rumus interpolasi:
   $$x_{new} = x_i + \lambda(x_{zi} - x_i)$$
   di mana $x_i$ adalah sampel minoritas asli, $x_{zi}$ adalah salah satu tetangga terdekatnya, dan $\lambda$ adalah angka acak antara 0 dan 1.
5. **Mengapa SMOTE Lebih Baik daripada Random Oversampling?**: Random Oversampling menduplikasi data minoritas asli secara berulang, yang menyebabkan model mengalami *overfitting* karena menghafal titik data yang sama. SMOTE menghasilkan variasi data baru yang realistis dalam batas-batas persebaran fitur minoritas asli.

### Catatan Penting: SMOTE Hanya untuk Data Latih
SMOTE **hanya boleh diterapkan pada data training saja**. Data testing harus tetap steril dan tidak dimanipulasi dengan cara apa pun. Hal ini untuk memastikan bahwa pengujian model mencerminkan kondisi klinis dunia nyata yang sesungguhnya.

In [ ]:

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Distribusi kelas sebelum SMOTE:")
print(y_train.value_counts())

print("\nDistribusi kelas sesudah SMOTE:")
print(y_train_smote.value_counts())

## 4. Metode & Eksperimen
Eksperimen ini mengevaluasi dan membandingkan tiga jenis algoritma pembelajaran mesin dengan karakteristik matematis yang berbeda:
1. **Decision Tree (Rule-based)**: Menggunakan pendekatan partisi ruang fitur secara berulang untuk meminimalkan ketidakmurnian (*impurity*).
2. **K-Nearest Neighbors (KNN) (Distance-based)**: Mengklasifikasikan data berdasarkan kesamaan jarak di ruang metrik.
3. **Gaussian Naive Bayes (Probabilistic-based)**: Berdasarkan probabilitas teoretis bersyarat dengan asumsi independensi fitur.

### Penjelasan Formula Matematis:

1. **Gini Impurity (Decision Tree)**:
   Mengukur tingkat keheterogenan suatu node. Algoritma mencari pemisahan fitur yang meminimalkan nilai Gini Impurity:
   $$Gini(D) = 1 - \sum_{i=1}^{C} (p_i)^2$$
   di mana $p_i$ adalah probabilitas kemunculan kelas $i$ pada node tersebut. Jika node berisi satu kelas saja (homogen), Gini Impurity bernilai 0.
2. **Jarak Euclidean (KNN)**:
   Jarak geometris antara dua titik data di ruang berdimensi-$N$:
   $$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$
3. **Teorema Bayes (Gaussian Naive Bayes)**:
   Menghitung probabilitas posterior suatu kelas berdasarkan probabilitas prior dan likelihood:
   $$P(C_k \mid x) = \frac{P(x \mid C_k) \cdot P(C_k)}{P(x)}$$
   Di mana $P(x \mid C_k)$ diasumsikan mengikuti distribusi Gaussian (Normal) untuk fitur kontinu.

### Skema Validasi dan GridSearchCV:
- **Pipeline Anti-Leakage [Sangat Krusial]**:
  Kami menyusun proses SMOTE dan pemodelan ke dalam sebuah `Pipeline` dari pustaka `imblearn.pipeline`. Saat melakukan pencarian parameter dengan *5-Fold Cross-Validation*, `Pipeline` memastikan bahwa proses resampling (SMOTE) **hanya diaplikasikan pada fold training di setiap iterasi**. Lipatan validasi pada iterasi tersebut tetap murni tanpa data sintetis. Hal ini mencegah *data leakage* teoretis, di mana informasi dari data validasi merembes ke proses latih.
- **Stratified 5-Fold Cross-Validation (`cv=5`)**:
  Pembagian data latih menjadi 5 bagian secara bergiliran. Kami memilih 5 lipatan karena porsi validasi per lipatan (sekitar 49 data) sudah stabil. Skema *Stratified* menjaga agar proporsi kelas target di tiap fold tetap konsisten dengan rasio populasi awal.
- **Scoring Metric `scoring='recall'`**:
  Kami mengarahkan pencarian hyperparameter terbaik untuk mengoptimalkan **Recall** pada kelas pasien meninggal (`Surv < 5 Years`). Secara klinis, recall didefinisikan sebagai:
  $$Recall = \frac{TP}{TP + FN}$$
  Mengoptimalkan recall berarti menekan nilai *False Negative* (FN) serendah mungkin, yang merupakan prioritas keselamatan pasien dalam diagnosis medis.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

grids = {
    'KNN': (KNeighborsClassifier(), {
        'model__n_neighbors': list(range(1, 21)), 
        'model__weights': ['uniform', 'distance']
    }),
    'Decision Tree': (DecisionTreeClassifier(random_state=42), {
        'model__max_depth': [3, 5, 7, None], 
        'model__min_samples_split': [2, 5, 10]
    }),
    'Naive Bayes': (GaussianNB(), {
        'model__var_smoothing': np.logspace(0, -9, num=50)
    })
}

best_models = {}
for name, (model, params) in grids.items():
    pipeline = Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])
    
    grid = GridSearchCV(pipeline, params, cv=5, scoring='recall')
    grid.fit(X_train_scaled, y_train)
    print(f"Parameter {name} terbaik:", grid.best_params_)
    best_models[name] = grid.best_estimator_

best_knn = best_models['KNN']
best_dt = best_models['Decision Tree']
best_nb = best_models['Naive Bayes']

## 5. Hasil & Analisis
pengujian performa model dilakukan menggunakan data uji (X_test_scaled dan y_test) untuk mengevaluasi kemampuan generalisasi model pada data yang tidak digunakan saat pelatihan.

##### Perulangan Model (for name, model in best_models.items():)

* Kode ini digunakan untuk mengotomatisasi proses evaluasi dengan melakukan perulangan (looping) pada seluruh model terbaik yang tersimpan di dalam dictionary best_models (Decision Tree, KNN, dan Naive Bayes).

* Tujuan: Efisiensi penulisan kode agar kelompok Anda tidak perlu menuliskan baris pengujian yang sama berulang kali untuk setiap algoritma, serta memastikan seluruh model diuji dalam kondisi lingkungan data uji yang sama persis (X_test_scaled).

 ##### Pembuatan Prediksi Kolektif (model.predict(X_test_scaled))

* Berfungsi untuk memerintahkan setiap model terbaik memprediksi status kelangsungan hidup pasien menggunakan data fitur uji yang telah distandarisasi (X_test_scaled).

* Tujuan: Menghasilkan nilai output prediksi (y_pred) yang murni bersumber dari kemampuan generalisasi masing-masing arsitektur algoritma terhadap data klinis baru 

##### Kustomisasi Label (target_names=['Surv >= 5 Years', 'Surv < 5 Years'])

* Parameter target_names berfungsi untuk mengubah representasi angka kode mentah target data (Angka 1 dan 2) menjadi label teks yang memiliki arti klinis jelas pada output cetakan.

In [ ]:
from sklearn.metrics import classification_report

for name, model in best_models.items():
    y_pred = model.predict(X_test_scaled)
    print(f"\n=== Laporan Klasifikasi {name} ===")
    print(classification_report(y_test, y_pred, target_names=['Surv >= 5 Years', 'Surv < 5 Years']))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['Blues', 'Greens', 'Oranges']

for ax, (name, model), color in zip(axes, best_models.items(), colors):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=color, ax=ax,
                xticklabels=['>= 5 Years', '< 5 Years'], yticklabels=['>= 5 Years', '< 5 Years'])
    ax.set_title(f'Confusion Matrix - {name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
plt.show()

### Analisis Kepercayaan Prediksi (Prediction Confidence)
Tingkat kepercayaan prediksi diukur berdasarkan nilai probabilitas posterior maksimum pada kelas yang diprediksi oleh masing-masing model klasifikasi. Evaluasi dilakukan untuk membandingkan rata-rata nilai kepercayaan pada prediksi yang benar dengan prediksi yang salah

In [ ]:
print("=== Rata-Rata Kepercayaan Prediksi (Test Set) ===\n")

for name, model in best_models.items():
    probs = model.predict_proba(X_test_scaled)
    y_pred = model.predict(X_test_scaled)
    
    confidences = np.max(probs, axis=1)
    
    correct = (y_pred == y_test)
    incorrect = (y_pred != y_test)
    
    print(f"[{name}]")
    print(f"Avg Confidence (BENAR): {np.mean(confidences[correct]):.4f}")
    print(f"Avg Confidence (SALAH): {np.mean(confidences[incorrect]):.4f}\n")

### Diskusi Perbandingan Hasil Model
Setelah kodenya dijalankan dengan benar (menggunakan Pipeline untuk mencegah kebocoran data/data leakage), berikut adalah hasil analisis dan diskusi kelompok kami:

1. **Akurasi Klasifikasi**:
   Akurasi model berkisar antara 56% sampai 66%. Rendahnya akurasi global ini dikarenakan dataset Haberman memiliki tingkat tumpang tindih (overlap) yang sangat tinggi pada ruang fitur. Secara medis, karakteristik klinis antara pasien yang bertahan hidup >= 5 tahun dan yang meninggal < 5 tahun sangat mirip pada 3 parameter yang ada (Usia, Tahun Operasi, dan Jumlah Kelenjar Getah Bening).

2. **Performa Deteksi Kelas Kritis (Recall Pasien Meninggal / Kelas Kritis)**:
   Secara klinis, kesalahan memprediksi pasien yang sebenarnya akan meninggal < 5 tahun sebagai pasien selamat (False Negative) adalah kesalahan fatal karena pasien tidak akan mendapatkan penanganan lanjutan yang intensif. Oleh karena itu, metrik utama yang kami optimalkan adalah **Recall** untuk kelas minoritas (`Surv < 5 Years`):
   - **Decision Tree** memberikan performa terbaik dengan Recall **44%** (F1-score 37%).
   - **KNN** memberikan performa menengah dengan Recall **19%** (F1-score 18%).
   - **Naive Bayes** memberikan performa terendah dengan Recall **12%** (F1-score 16%).

3. **Analisis Kepercayaan Prediksi (Prediction Confidence)**:
   - **Decision Tree**: Rata-rata tingkat kepercayaan sangat tinggi, baik saat benar (95.23%) maupun saat salah (92.06%). Hal ini wajar bagi Decision Tree karena kecenderungannya membuat keputusan deterministik di leaf nodes, meskipun pembatasan kedalaman (`max_depth=7`) telah membatasi overconfidence ekstrem.
   - **KNN**: Menghasilkan tingkat kepercayaan yang proporsional (87.38% saat benar, 81.04% saat salah). Nilai ini dihitung dari proporsi voting tetangga ($K=6$) yang diperberat oleh jarak (`distance`).
   - **Naive Bayes**: Menghasilkan kepercayaan yang lebih rendah dan moderat (62.61% saat benar, 66.86% saat salah). Hal ini karena Naive Bayes memodelkan probabilitas kontinu secara halus (Gaussian), sehingga tidak menghasilkan estimasi probabilitas ekstrem.

**Kesimpulan Pemilihan Model**: Kelompok kami merekomendasikan **Decision Tree** dan **K-Nearest Neighbors (KNN)** untuk penulisan laporan final paper IEEE. Kedua model ini terbukti memiliki tingkat sensitivitas (Recall) yang lebih baik dalam mendeteksi kelas kritis/pasien berisiko meninggal dibandingkan Naive Bayes.

## 6. Kesimpulan

1. Eksperimen perbandingan algoritma Decision Tree, K-Nearest Neighbors (KNN), dan Naive Bayes telah dilakukan untuk mengklasifikasikan status kelangsungan hidup pasien kanker payudara pada dataset Haberman.
2. Ketidakseimbangan kelas pada dataset diatasi menggunakan teknik SMOTE pada data pelatihan untuk menghasilkan distribusi kelas latih yang seimbang.
3. Penentuan parameter optimal dari masing-masing model dilakukan melalui metode GridSearchCV dengan 5-Fold Stratified Cross-Validation.
4. Berdasarkan hasil pengujian pada data uji, Decision Tree dan K-Nearest Neighbors (KNN) diidentifikasi sebagai dua metode terbaik. Keduanya direkomendasikan untuk laporan akhir tugas besar karena menunjukkan performa klasifikasi yang lebih sensitif terhadap kelas minoritas berisiko tinggi (Class 2) dibandingkan dengan Naive Bayes.